<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase43_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 43 — frozen-method synthetic holdout and no-retuning audit

 The notebook mounts Google Drive, finds the verified Phase 39 source ledger and Phase 42 outputs by SHA-256, and writes a timestamped folder to `MyDrive/ICHI2027/Phase43/`.

**Purpose:** freeze the exact Phase 42 method, generate a new deterministic synthetic holdout, seal the truth manifest before detection, run the unchanged detector, and quantify event-level and cohort-level consequences. This remains developer-authored synthetic engineering evidence—not independent validation, official OMOP/MI-CDM ETL, Arm A, frozen TFL-v15 Arm C, or the full A/B/C/D experiment.


In [1]:
# CODE CELL 1/8 — Drive, constants, local staging
import os,re,csv,json,hashlib,sqlite3,shutil,tempfile,zipfile,time,random
import datetime as dt
from pathlib import Path
from copy import deepcopy
from collections import Counter,defaultdict

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='43_frozen_method_synthetic_holdout_no_retuning_audit'
EXPECTED_P39_DB='0c9507d8fcf5986e9763d473bf0b342a1c4daaad6c87ee04915986518f5c2384'
EXPECTED_P42_SUMMARY='fe20f91ced49f1d86f22cbdc44268ffbc078b6507c8442271d9fe7d73214f787'
EXPECTED_P42_METHOD_FILE='490657f569f36fec6dd7c4b793556490a4c1318249fbb9111999f3150ab149a8'
EXPECTED_P42_METHOD_CONFIG='5bac736739cffa44b1ebbc2ced2886f7e267887a4e78a1b1ba8e012651616902'
EXPECTED_GATES='4c1febebd83ee969f9102499ddd09fba6542092c202fcfd7d98015797f1f454b'
EXPECTED_CONTRACT='d80e7ede309fedca3e95341b299ae4756c9a6796ea2148c4abc0479f8ee235da'
HOLDOUT_SEED=430930

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''): h.update(b)
    return h.hexdigest()

if os.environ.get('PHASE43_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE43_LOCAL_DRIVE_ROOT']).expanduser().resolve();ROOTS=[MYDRIVE]
    print('Offline Drive mirror:',MYDRIVE)
else:
    try: from google.colab import drive
    except ImportError as e: raise RuntimeError('Use Colab or set PHASE43_LOCAL_DRIVE_ROOT.') from e
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive'); assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    shared=Path('/content/drive/Shareddrives')
    if shared.is_dir(): ROOTS.append(shared)

hint=os.environ.get('PHASE43_DRIVE_FOLDER_HINT','').strip()
if hint:
    p=Path(hint).expanduser(); p=p if p.is_absolute() else MYDRIVE/p
    if not p.is_dir(): raise FileNotFoundError('Drive hint does not exist: '+hint)
    ROOTS=[p.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase43_'))
INPUT=WORK/'inputs';INPUT.mkdir();STAGE=WORK/'outputs';STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE43_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase43')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
print('Output:',OUT)


Mounted at /content/drive
Output: /content/drive/MyDrive/ICHI2027/Phase43/run_20261001T011721_008153Z


## 1 · Automatic cryptographic input discovery
No upload widget is used. Duplicate filenames such as `(1)` are allowed; only checksum-matching copies are accepted.


In [2]:
# CODE CELL 2/8 — Verify Phase 39/42 inputs
REQUIRED={
 'p39_db':('ICHI2027_Phase39_SYNTHETIC_SOURCE_LEDGER','.sqlite',EXPECTED_P39_DB),
 'p42_summary':('ICHI2027_Phase42_SHAREABLE','.json',EXPECTED_P42_SUMMARY),
 'p42_method':('ICHI2027_Phase42_METHOD_SPEC','.json',EXPECTED_P42_METHOD_FILE),
 'p42_gates':('ICHI2027_Phase42_ARM_A_GATES_UNCHANGED','.csv',EXPECTED_GATES),
 'p42_contract':('ICHI2027_Phase42_LOGICAL_CONTRACT_UNCHANGED','.csv',EXPECTED_CONTRACT),
}
def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None
hits={k:[] for k in REQUIRED}
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for f in files:
            for k,(stem,ext,digest) in REQUIRED.items():
                if name_matches(f,stem,ext): hits[k].append(Path(here)/f)
missing=[k for k,v in hits.items() if not v]
if missing: raise FileNotFoundError('Missing required Drive files; no manual upload fallback: '+', '.join(missing))
P={}
for k,(stem,ext,digest) in REQUIRED.items():
    good=[p for p in hits[k] if sha256(p)==digest]
    if not good: raise RuntimeError('No approved checksum copy for '+k+'; expected '+digest)
    P[k]=sorted(good,key=str)[0]
M42=json.loads(P['p42_summary'].read_text('utf-8'))
METHOD=json.loads(P['p42_method'].read_text('utf-8'))
method_blob=json.dumps(METHOD,sort_keys=True,separators=(',',':')).encode('utf-8')
assert hashlib.sha256(method_blob).hexdigest()==EXPECTED_P42_METHOD_CONFIG
assert M42['method_config_sha256']==EXPECTED_P42_METHOD_CONFIG
assert M42['phase']=='42_internal_synthetic_policy_algorithm_and_ablations'
assert M42['armA_unapproved_original_gates']==10
assert M42['official_omop_etl_executed'] is False and M42['official_micdm_load_executed'] is False
assert M42['independently_blinded_external_validation'] is False
assert M42['frozen_original_generic_tfl_armC_executed'] is False
with P['p42_gates'].open(newline='',encoding='utf-8') as f:GATES=list(csv.DictReader(f))
with P['p42_contract'].open(newline='',encoding='utf-8') as f:CONTRACT=list(csv.DictReader(f))
assert len(GATES)==10 and all(x['official_validation_passed']=='False' for x in GATES)
assert len(CONTRACT)==8
DB_LOCAL=INPUT/'immutable_source.sqlite';shutil.copyfile(P['p39_db'],DB_LOCAL);assert sha256(DB_LOCAL)==EXPECTED_P39_DB
print('PASS: verified Phase 39 source and frozen Phase 42 method/evidence files.')
print('Method config SHA-256:',EXPECTED_P42_METHOD_CONFIG)


PASS: verified Phase 39 source and frozen Phase 42 method/evidence files.
Method config SHA-256: 5bac736739cffa44b1ebbc2ced2886f7e267887a4e78a1b1ba8e012651616902


## 2 · Reconstruct the frozen Phase 42 method and immutable synthetic source
Phase 43 does not add reason codes, alter thresholds, or change action mappings.


In [3]:
# CODE CELL 3/8 — Read-only source + frozen method
c=sqlite3.connect('file:'+str(DB_LOCAL)+'?mode=ro',uri=True)
assert c.execute('PRAGMA integrity_check').fetchone()[0]=='ok';assert c.execute('PRAGMA foreign_key_check').fetchall()==[]
c.row_factory=sqlite3.Row
STUDIES={r['study_ref']:dict(r) for r in c.execute('SELECT * FROM source_study')}
DEVICES={r['device_ref']:dict(r) for r in c.execute('SELECT * FROM source_device')}
PROVS={r['obs_ref']:dict(r) for r in c.execute('SELECT * FROM source_observation_provenance')}
SOURCE=[]
for raw in c.execute('SELECT * FROM source_observation ORDER BY obs_ref'):
    d=dict(raw);dev=DEVICES[d['device_ref']];d.update(algorithm_name=dev['algorithm_name'],algorithm_version=dev['algorithm_version'])
    assert PROVS[d['obs_ref']]['provenance_ref']==d['provenance_ref'];SOURCE.append(d)
c.close()
assert len(SOURCE)==148 and len(STUDIES)==38 and len(DEVICES)==19
assert Counter(r['trace_state'] for r in SOURCE)=={'PASS':148}
RULE_GROUPS=METHOD['groups'];CRITICAL=set(METHOD['critical_exclude_reason_codes']);REVIEW=set(METHOD['audit_review_reason_codes'])
ABS_TOLERANCE_MM3=float(METHOD['absolute_value_tolerance_mm3']);FULL_GROUPS=METHOD['ablations']['05_FULL']
FULL_CODES={code for g in FULL_GROUPS for code in RULE_GROUPS[g]}
EXPECTED_CODES={'ANATOMY_LATERALITY_MISMATCH','UNIT_MISSING_OR_INCOMPATIBLE','VALUE_MISMATCH','PERSON_LINK_MISMATCH','STUDY_TIMEPOINT_LINK_MISMATCH','MODEL_IDENTITY_LOSS','PROVENANCE_LINK_LOSS','TRACE_STATE_LOSS','DUPLICATE_OBSERVATION','DROPPED_OBSERVATION','UNEXPECTED_OBSERVATION'}
assert FULL_CODES==EXPECTED_CODES and CRITICAL|REVIEW==EXPECTED_CODES and not CRITICAL&REVIEW
print('PASS: full Phase 42 rule set frozen:',len(FULL_CODES),'reason codes.')


PASS: full Phase 42 rule set frozen: 11 reason codes.


## 3 · Frozen source/target queries, detector and policy
The detector receives only source and projected records; the truth registry is not passed into detection.


In [4]:
# CODE CELL 4/8 — Frozen queries and detector
FIELDS=['obs_ref','person_ref','study_ref','biomarker_code','anatomy_source_code','laterality','timepoint','effective_utc','value_mm3','unit_code','algorithm_name','algorithm_version','trace_state','trace_version','device_ref','provenance_ref']
DDL="""
CREATE TABLE projected_study(study_ref TEXT,person_ref TEXT,study_started TEXT);
CREATE TABLE projected_feature(obs_ref TEXT,person_ref TEXT,study_ref TEXT,biomarker_code TEXT,anatomy_source_code TEXT,laterality TEXT,timepoint TEXT,effective_utc TEXT,value_mm3 REAL,unit_code TEXT,algorithm_name TEXT,algorithm_version TEXT,trace_state TEXT,trace_version TEXT,device_ref TEXT,provenance_ref TEXT);
"""
INSERT='INSERT INTO projected_feature('+','.join(FIELDS)+') VALUES('+','.join('?' for _ in FIELDS)+')'
SQL_PROXY="""
WITH observed AS(
 SELECT f.*,s.person_ref AS linked_person,s.study_started AS linked_started,
 CASE WHEN f.unit_code='mm3' THEN f.value_mm3 WHEN f.unit_code='cm3' THEN f.value_mm3*1000.0 ELSE NULL END AS normalized_mm3
 FROM projected_feature f LEFT JOIN projected_study s ON f.study_ref=s.study_ref WHERE f.biomarker_code='hippocampus-volume'
),valid AS(
 SELECT *,CASE WHEN linked_person=person_ref AND linked_started=effective_utc AND normalized_mm3>0 AND provenance_ref IS NOT NULL
 AND ((laterality='LEFT' AND anatomy_source_code='SYN-HIP-LEFT') OR (laterality='RIGHT' AND anatomy_source_code='SYN-HIP-RIGHT'))
 AND timepoint IN ('baseline','12-month') THEN 1 ELSE 0 END AS valid_flag FROM observed
),slots AS(
 SELECT person_ref,timepoint,laterality,COUNT(*) AS n,SUM(valid_flag) AS nvalid,
 SUM(CASE WHEN trace_state IS NULL OR trace_state='REVIEW_REQUIRED' THEN 1 ELSE 0 END) AS unqualified
 FROM valid GROUP BY person_ref,timepoint,laterality
),eligible AS(
 SELECT person_ref FROM slots GROUP BY person_ref HAVING COUNT(*)=4 AND MIN(n)=1 AND MAX(n)=1 AND SUM(nvalid)=4
)
SELECT person_ref FROM eligible ORDER BY person_ref
"""
SQL_QUALIFIED=SQL_PROXY.replace('AND SUM(nvalid)=4','AND SUM(nvalid)=4 AND SUM(unqualified)=0')
SQL_VALUE_ONLY="""
SELECT person_ref FROM projected_feature WHERE biomarker_code='hippocampus-volume' AND value_mm3>0
AND laterality IN ('LEFT','RIGHT') AND timepoint IN ('baseline','12-month')
GROUP BY person_ref HAVING COUNT(DISTINCT timepoint||'/'||laterality)=4 ORDER BY person_ref
"""
def query(rows,sql):
    cx=sqlite3.connect(':memory:')
    try:
        cx.executescript(DDL);cx.executemany('INSERT INTO projected_study VALUES (?,?,?)',[(s['study_ref'],s['person_ref'],s['study_started']) for s in STUDIES.values()])
        cx.executemany(INSERT,[tuple(r.get(k) for k in FIELDS) for r in rows]);return {z[0] for z in cx.execute(sql)}
    finally:cx.close()
SOURCE_PROXY=query(SOURCE,SQL_PROXY);SOURCE_QUAL=query(SOURCE,SQL_QUALIFIED);assert SOURCE_PROXY==SOURCE_QUAL and len(SOURCE_PROXY)==15
ALL_PEOPLE=sorted({r['person_ref'] for r in SOURCE});ELIGIBLE=sorted(SOURCE_PROXY);INELIGIBLE=sorted(set(ALL_PEOPLE)-SOURCE_PROXY)
HIP={p:[r['obs_ref'] for r in SOURCE if r['person_ref']==p and r['biomarker_code']=='hippocampus-volume'] for p in ALL_PEOPLE}

def detect(ref,target):
    src={x['obs_ref']:x for x in ref};assert len(src)==len(ref);tgt=defaultdict(list)
    for r in target:tgt[r['obs_ref']].append(r)
    alerts=set();normalized=set()
    for obs,s in src.items():
        got=tgt.get(obs,[])
        if len(got)!=1:
            alerts.add((obs,'DUPLICATE_OBSERVATION' if len(got)>1 else 'DROPPED_OBSERVATION'))
            if not got:continue
        t=got[0]
        if t['person_ref']!=s['person_ref']:alerts.add((obs,'PERSON_LINK_MISMATCH'))
        if any(t[k]!=s[k] for k in ('biomarker_code','anatomy_source_code','laterality')):alerts.add((obs,'ANATOMY_LATERALITY_MISMATCH'))
        if (any(t[k]!=s[k] for k in ('study_ref','timepoint','effective_utc')) or t['study_ref'] not in STUDIES or STUDIES[t['study_ref']]['person_ref']!=t['person_ref'] or STUDIES[t['study_ref']]['study_started']!=t['effective_utc']):alerts.add((obs,'STUDY_TIMEPOINT_LINK_MISMATCH'))
        unit=t['unit_code'];v=t['value_mm3']
        if unit not in {'mm3','cm3'} or not isinstance(v,(int,float)) or v<=0:alerts.add((obs,'UNIT_MISSING_OR_INCOMPATIBLE'))
        elif abs(float(v)*(1000.0 if unit=='cm3' else 1.0)-float(s['value_mm3']))>ABS_TOLERANCE_MM3:alerts.add((obs,'VALUE_MISMATCH'))
        elif unit=='cm3' and s['unit_code']=='mm3':normalized.add(obs)
        if t['algorithm_name']!=s['algorithm_name'] or t['algorithm_version']!=s['algorithm_version']:alerts.add((obs,'MODEL_IDENTITY_LOSS'))
        if t['trace_state']!=s['trace_state'] or t['trace_version']!=s['trace_version']:alerts.add((obs,'TRACE_STATE_LOSS'))
        if t['provenance_ref']!=s['provenance_ref'] or t['device_ref']!=s['device_ref']:alerts.add((obs,'PROVENANCE_LINK_LOSS'))
    for obs in set(tgt)-set(src):alerts.add((obs,'UNEXPECTED_OBSERVATION'))
    return alerts,normalized

def decisions(events,normalizations):
    per=defaultdict(set)
    for obs,reason in events:per[obs].add(reason)
    out={}
    for obs,reasons in per.items():
        if reasons&CRITICAL:out[obs]='EXCLUDE'
        elif reasons&REVIEW:out[obs]='REVIEW'
        else:raise AssertionError('Unclassified reason '+str(reasons))
    for obs in normalizations:
        if obs not in out:out[obs]='NORMALIZE'
    return out
assert not detect(SOURCE,SOURCE)[0]
print('Frozen reference:',len(ALL_PEOPLE),'people;',len(ELIGIBLE),'proxy eligible;',len(INELIGIBLE),'proxy ineligible.')


Frozen reference: 19 people; 15 proxy eligible; 4 proxy ineligible.


## 4 · Generate and seal a new holdout before detection
A new deterministic seed, alternate mutation parameterizations, the previously unused `UNEXPECTED_OBSERVATION`, valid unit normalization, and compound faults are used. The truth registry is SHA-256 sealed before any detector call in the next cell.


In [5]:
# CODE CELL 5/8 — Holdout generator and pre-detection truth seal
RNG=random.Random(HOLDOUT_SEED);HOLDOUT=[];TRUTH={};NORMS={}
def row_by_id(rows,obs):
    hit=[r for r in rows if r['obs_ref']==obs];assert len(hit)==1,obs;return hit[0]
def new_case(kind,person):
    cid=f'holdout_{len(HOLDOUT):04d}';f={'case_id':cid,'kind':kind,'person_ref':person,'reference':deepcopy(SOURCE),'projected':deepcopy(SOURCE)}
    HOLDOUT.append(f);TRUTH[cid]=set();NORMS[cid]=set();return f
def mark(f,obs,reason):TRUTH[f['case_id']].add((obs,reason))
def inject(f,kind,obs):
    if kind=='ANATOMY_LATERALITY_MISMATCH':
        t=row_by_id(f['projected'],obs);t['anatomy_source_code']='SYN-HIP-RIGHT' if t['anatomy_source_code']=='SYN-HIP-LEFT' else 'SYN-HIP-LEFT';mark(f,obs,kind)
    elif kind=='UNIT_MISSING_OR_INCOMPATIBLE':row_by_id(f['projected'],obs).__setitem__('unit_code','mL');mark(f,obs,kind)
    elif kind=='VALUE_MISMATCH':
        t=row_by_id(f['projected'],obs);t['value_mm3']=float(t['value_mm3'])*1.01+1.0;mark(f,obs,kind)
    elif kind=='PERSON_LINK_MISMATCH':
        t=row_by_id(f['projected'],obs);t['person_ref']=next(p for p in ALL_PEOPLE if p!=t['person_ref']);mark(f,obs,'PERSON_LINK_MISMATCH');mark(f,obs,'STUDY_TIMEPOINT_LINK_MISMATCH')
    elif kind=='STUDY_TIMEPOINT_LINK_MISMATCH':row_by_id(f['projected'],obs).__setitem__('effective_utc','2099-01-01T00:00:00Z');mark(f,obs,kind)
    elif kind=='MODEL_IDENTITY_LOSS':row_by_id(f['projected'],obs).__setitem__('algorithm_name',None);mark(f,obs,kind)
    elif kind=='PROVENANCE_LINK_LOSS':row_by_id(f['projected'],obs).__setitem__('device_ref','Device/phase43-missing');mark(f,obs,kind)
    elif kind=='TRACE_STATE_LOSS':row_by_id(f['reference'],obs).__setitem__('trace_state','REVIEW_REQUIRED');row_by_id(f['projected'],obs).__setitem__('trace_version',None);mark(f,obs,kind)
    elif kind=='DUPLICATE_OBSERVATION':f['projected'].append(deepcopy(row_by_id(f['projected'],obs)));mark(f,obs,kind)
    elif kind=='DROPPED_OBSERVATION':f['projected']=[r for r in f['projected'] if r['obs_ref']!=obs];mark(f,obs,kind)
    elif kind=='UNEXPECTED_OBSERVATION':
        t=deepcopy(row_by_id(f['projected'],obs));t['obs_ref']='Observation/phase43-unexpected-'+f['case_id'];f['projected'].append(t);mark(f,t['obs_ref'],kind)
    else:raise AssertionError(kind)

for p in ALL_PEOPLE:new_case('CLEAN',p)
FAMILIES=['ANATOMY_LATERALITY_MISMATCH','UNIT_MISSING_OR_INCOMPATIBLE','VALUE_MISMATCH','PERSON_LINK_MISMATCH','STUDY_TIMEPOINT_LINK_MISMATCH','MODEL_IDENTITY_LOSS','PROVENANCE_LINK_LOSS','TRACE_STATE_LOSS','DUPLICATE_OBSERVATION','DROPPED_OBSERVATION','UNEXPECTED_OBSERVATION']
for p in ELIGIBLE:
    for fam in FAMILIES:
        f=new_case(fam,p);inject(f,fam,RNG.choice(HIP[p]))
for p in ELIGIBLE:
    f=new_case('VALID_CM3_NORMALIZATION',p);obs=RNG.choice(HIP[p]);t=row_by_id(f['projected'],obs);t['unit_code']='cm3';t['value_mm3']=float(t['value_mm3'])/1000.0;NORMS[f['case_id']].add(obs)
for i in range(30):
    p=ELIGIBLE[i%len(ELIGIBLE)];obs1,obs2=RNG.sample(HIP[p],2);f=new_case('COMPOUND_HOLDOUT',p)
    if i%2==0:inject(f,'VALUE_MISMATCH',obs1);inject(f,'PROVENANCE_LINK_LOSS',obs2)
    else:inject(f,'UNIT_MISSING_OR_INCOMPATIBLE',obs1);inject(f,'DROPPED_OBSERVATION',obs2)
SEAL_ROWS=[{'case_id':f['case_id'],'kind':f['kind'],'person_ref':f['person_ref'],'truth_events':sorted([list(x) for x in TRUTH[f['case_id']]]),'expected_normalizations':sorted(NORMS[f['case_id']]),'projected_rows':len(f['projected'])} for f in HOLDOUT]
SEAL_BLOB=json.dumps(SEAL_ROWS,sort_keys=True,separators=(',',':')).encode('utf-8');HOLDOUT_TRUTH_SEAL_SHA256=hashlib.sha256(SEAL_BLOB).hexdigest()
assert len(HOLDOUT)==229
print('SEALED BEFORE DETECTION:',len(HOLDOUT),'cases')
print('Truth seal SHA-256:',HOLDOUT_TRUTH_SEAL_SHA256)


SEALED BEFORE DETECTION: 229 cases
Truth seal SHA-256: 6952711786b4b99d37204dbbc70762caabfbdb852d0251be5ef231efaf72609b


## 5 · Run unchanged detector against the sealed holdout
These are internal regression counts only; they are not unbiased sensitivity/specificity estimates.


In [6]:
# CODE CELL 6/8 — Event-level holdout audit
EVENT_ROWS=[];CASE_ROWS=[];total_truth=total_tp=total_fp=0;norm_expected=norm_detected=0
for f in HOLDOUT:
    cid=f['case_id'];alerts,norm=detect(f['reference'],f['projected']);truth=TRUTH[cid];expected=NORMS[cid]
    tp=alerts&truth;fp=alerts-truth;fn=truth-alerts
    total_truth+=len(truth);total_tp+=len(tp);total_fp+=len(fp);norm_expected+=len(expected);norm_detected+=len(norm&expected)
    CASE_ROWS.append({'case_id':cid,'kind':f['kind'],'person_ref':f['person_ref'],'projected_rows':len(f['projected']),'sealed_truth_events':len(truth),'detected_truth_events':len(tp),'missed_truth_events':len(fn),'unexpected_alert_events':len(fp),'expected_normalizations':len(expected),'detected_expected_normalizations':len(norm&expected),'truth_seal_sha256':HOLDOUT_TRUTH_SEAL_SHA256,'same_project_authored':'True','independently_blinded':'False'})
    for obs,reason in sorted(truth|alerts):EVENT_ROWS.append({'case_id':cid,'kind':f['kind'],'person_ref':f['person_ref'],'obs_ref':obs,'reason_code':reason,'sealed_truth':'True' if (obs,reason) in truth else 'False','detected':'True' if (obs,reason) in alerts else 'False','matched':'True' if (obs,reason) in tp else 'False','false_alert':'True' if (obs,reason) in fp else 'False','missed':'True' if (obs,reason) in fn else 'False'})
assert total_tp==total_truth and total_fp==0
assert norm_detected==norm_expected==15
assert hashlib.sha256(json.dumps(SEAL_ROWS,sort_keys=True,separators=(',',':')).encode('utf-8')).hexdigest()==HOLDOUT_TRUTH_SEAL_SHA256
assert hashlib.sha256(json.dumps(METHOD,sort_keys=True,separators=(',',':')).encode('utf-8')).hexdigest()==EXPECTED_P42_METHOD_CONFIG
print('PASS: matched authored events',total_tp,'/',total_truth,'; unexpected alerts',total_fp,'; normalizations',norm_detected,'/',norm_expected)
print('Boundary: developer-authored synthetic holdout; not independent validation.')


PASS: matched authored events 240 / 240 ; unexpected alerts 0 ; normalizations 15 / 15
Boundary: developer-authored synthetic holdout; not independent validation.


## 6 · Frozen policy actions and person-level cohort consequences


In [7]:
# CODE CELL 7/8 — Policy and cohort comparison
def cmp(a,b):
    u=a|b;return {'source_n':len(a),'target_n':len(b),'false_inclusions':len(b-a),'false_exclusions':len(a-b),'jaccard':round(len(a&b)/len(u),6) if u else 1.0}
ACTION_ROWS=[];COHORT_ROWS=[]
for f in HOLDOUT:
    alerts,norm=detect(f['reference'],f['projected']);action=decisions(alerts,norm)
    for obs,state in sorted(action.items()):ACTION_ROWS.append({'case_id':f['case_id'],'kind':f['kind'],'person_ref':f['person_ref'],'obs_ref':obs,'action':state,'reason_codes':';'.join(sorted(r for o,r in alerts if o==obs)),'official_target':'False','developer_synthetic':'True'})
    unqualified=[r for r in f['projected'] if action.get(r['obs_ref'])!='EXCLUDE'];qualified=[r for r in unqualified if action.get(r['obs_ref'])!='REVIEW']
    ref_proxy=query(f['reference'],SQL_PROXY);ref_qual=query(f['reference'],SQL_QUALIFIED)
    outcomes=[('STRICT_PROJECTED_DIAGNOSTIC',ref_proxy,query(f['projected'],SQL_PROXY)),('VALUE_ONLY_UNSAFE_DIAGNOSTIC',ref_proxy,query(f['projected'],SQL_VALUE_ONLY)),('FROZEN_POLICY_PROXY',ref_proxy,query(unqualified,SQL_VALUE_ONLY)),('FROZEN_POLICY_QUALIFIED',ref_qual,query(qualified,SQL_VALUE_ONLY))]
    for label,ref,tgt in outcomes:COHORT_ROWS.append({'case_id':f['case_id'],'kind':f['kind'],'person_ref':f['person_ref'],'cohort_policy':label,'source_people_total':19,'source_rows_total':148,**cmp(ref,tgt),'clinical_MCI_attested':'False','official_target':'False'})
assert len(COHORT_ROWS)==len(HOLDOUT)*4 and len(GATES)==10
changed=sum(1 for r in COHORT_ROWS if int(r['false_inclusions']) or int(r['false_exclusions']))
print('Cohort comparisons:',len(COHORT_ROWS),'| membership-changing comparisons:',changed)


Cohort comparisons: 916 | membership-changing comparisons: 465


## 7 · Save public-safe outputs back to Drive
All outputs are staged locally, hashed, copied to a new Drive run folder, then re-hashed after the copy.


In [8]:
# CODE CELL 8/8 — Save CSV/JSON/ZIP and verify Drive read-back
def save_csv(name,rows):
    assert rows;p=STAGE/name
    with p.open('w',newline='',encoding='utf-8') as f:w=csv.DictWriter(f,fieldnames=list(rows[0].keys()));w.writeheader();w.writerows(rows)
    return p
files=[save_csv('ICHI2027_Phase43_HOLDOUT_CASES.csv',CASE_ROWS),save_csv('ICHI2027_Phase43_HOLDOUT_EVENTS.csv',EVENT_ROWS),save_csv('ICHI2027_Phase43_POLICY_ACTIONS.csv',ACTION_ROWS),save_csv('ICHI2027_Phase43_COHORT_CONSEQUENCES.csv',COHORT_ROWS),save_csv('ICHI2027_Phase43_ARM_A_GATES_UNCHANGED.csv',GATES),save_csv('ICHI2027_Phase43_LOGICAL_CONTRACT_UNCHANGED.csv',CONTRACT)]
lock={'project':PROJECT,'phase':PHASE,'phase42_method_file_sha256':EXPECTED_P42_METHOD_FILE,'phase42_method_config_sha256':EXPECTED_P42_METHOD_CONFIG,'method_version':METHOD['method_version'],'holdout_seed':HOLDOUT_SEED,'holdout_truth_seal_sha256':HOLDOUT_TRUTH_SEAL_SHA256,'holdout_cases':len(HOLDOUT),'reason_codes_frozen':sorted(FULL_CODES),'absolute_value_tolerance_mm3':ABS_TOLERANCE_MM3,'retuning_after_holdout_generation':False,'independent_external_author':False,'official_target_schema_used':False}
lockp=STAGE/'ICHI2027_Phase43_METHOD_LOCK.json';lockp.write_text(json.dumps(lock,sort_keys=True,indent=2)+'\n',encoding='utf-8');files.append(lockp)
readme=STAGE/'ICHI2027_Phase43_README.txt';readme.write_text('PHASE 43: frozen Phase 42 method on a new developer-authored synthetic holdout.\nTruth was SHA-256 sealed before detection; method configuration was unchanged.\nThis is NOT independent validation, official OMOP/MI-CDM ETL, Arm A, frozen generic TFL Arm C, or full A/B/C/D results.\nNo real MRI rows, clinical MCI labels, private Athena data, reviewed concept IDs, or real DICOM UIDs were used.\nThe synthetic source proxy cohort remains 15/19 and all ten original Arm A gates remain open.\n',encoding='utf-8');files.append(readme)
summary={'project':PROJECT,'phase':PHASE,'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),'scope':'PUBLIC_SYNTHETIC_FROZEN_METHOD_HOLDOUT_DEVELOPER_AUTHORED','input_phase39_db_sha256':sha256(DB_LOCAL),'input_phase42_summary_sha256':sha256(P['p42_summary']),'input_phase42_method_file_sha256':sha256(P['p42_method']),'phase42_method_config_sha256':EXPECTED_P42_METHOD_CONFIG,'method_version':METHOD['method_version'],'holdout_seed':HOLDOUT_SEED,'holdout_truth_seal_sha256':HOLDOUT_TRUTH_SEAL_SHA256,'holdout_cases':len(HOLDOUT),'holdout_authored_fault_events':total_truth,'holdout_matched_authored_fault_events':total_tp,'holdout_unexpected_alert_events':total_fp,'holdout_valid_normalizations':norm_expected,'holdout_detected_valid_normalizations':norm_detected,'person_level_cohort_comparisons':len(COHORT_ROWS),'method_retuned_after_holdout_generation':False,'independently_blinded_external_validation':False,'official_omop_etl_executed':False,'official_micdm_load_executed':False,'frozen_original_generic_tfl_armC_executed':False,'real_mri_rows_processed':0,'full_A_B_C_D_evaluation':False,'clinical_mci_attested':False,'synthetic_people':19,'synthetic_proxy_positive':15,'synthetic_proxy_negative':4,'armA_status':'BLOCKED_OFFICIAL_ARM_A_NOT_EXECUTED','armA_unapproved_original_gates':len(GATES),'artifact_sha256':{p.name:sha256(p) for p in files},'warning':'No-retuning developer-authored synthetic holdout evidence is not independent validation or an official target-schema experiment.'}
summaryp=STAGE/'ICHI2027_Phase43_SHAREABLE.json';summaryp.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8');public=files+[summaryp]
zipp=STAGE/'ICHI2027_Phase43_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zipp,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in public:z.write(p,p.name)
OUT.mkdir(parents=True,exist_ok=False)
for p in public+[zipp]:
    target=OUT/p.name
    for attempt in range(3):
        try:shutil.copyfile(p,target);assert sha256(target)==sha256(p);break
        except (OSError,AssertionError) as e:
            if attempt==2:raise RuntimeError('Drive save/read-back failed for '+p.name) from e
            time.sleep(1+attempt)
print('SUCCESS:',len(public)+1,'files saved and read-back SHA-256 verified.')
print('Output folder:',OUT)
print('Shareable JSON:',OUT/summaryp.name)
print('Public ZIP:',OUT/zipp.name)


SUCCESS: 10 files saved and read-back SHA-256 verified.
Output folder: /content/drive/MyDrive/ICHI2027/Phase43/run_20261001T011721_008153Z
Shareable JSON: /content/drive/MyDrive/ICHI2027/Phase43/run_20261001T011721_008153Z/ICHI2027_Phase43_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase43/run_20261001T011721_008153Z/ICHI2027_Phase43_PUBLIC_OUTPUTS.zip


## Next material scientific gate
If Phase 43 passes, the internal synthetic development track should be treated as frozen. The next material advance is to **clear the ten Arm A evidence gates and run the official target loader**, execute the **unmodified frozen generic TFL v15** as the real Arm C comparator, or obtain **independently authored/blinded challenge cases**. More self-authored synthetic cases would add little scientific weight.
